# Análise de Fairness Estratificada — Modelos de Prognóstico de Dengue

Avalia se a performance preditiva dos modelos treinados é equânime entre subgrupos
populacionais relevantes. Métricas calculadas com IC 95% via bootstrap não paramétrico
(1.000 reamostragens, `random_state=42`).

**Conjunto de teste:** 2024 (n = 160.534; 5.295 óbitos; prevalência = 3,3%)  
**Threshold fixo:** 0,5  
**Limiar de alerta:** ΔAUPRC > 0,05 entre melhor e pior subgrupo

**Variáveis de estratificação:**
1. Faixa etária (`age_years`)
2. Sexo (`CS_SEXO`)
3. Raça/cor (`CS_RACA`)
4. Escolaridade (`CS_ESCOL_N`)
5. Gestação (`CS_GESTANT`)
6. Região geográfica (`SG_UF`)

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score

# Caminhos relativos ao notebook em notebooks/06_fairness/
DATA_DIR   = '../../data/features/baseline'
OUTPUT_MET = '../../output/metricas'
OUTPUT_PLT = '../../output/plots'
os.makedirs(OUTPUT_PLT, exist_ok=True)

# Parâmetros globais
THRESHOLD   = 0.5
N_BOOT      = 2000
RANDOM_STATE = 42
MIN_EVENTOS  = 30  # mínimo de eventos positivos por subgrupo para inclusão

LABEL_MAP = {
    'logistic_regression_baseline_tuned': 'LR (tuned)',
    'lightgbm_baseline_tuned':            'LightGBM (tuned)',
    'xgboost_baseline_tuned':             'XGBoost (tuned)',
    'random_forest_baseline':             'RF',
    'decision_tree_baseline':       'DT',
}

print(f'{len(LABEL_MAP)} modelos configurados.')

In [ ]:
# Carregar X_test com variáveis originais (antes do pré-processamento)
X_test = pd.read_parquet(os.path.join(DATA_DIR, 'X_test.parquet'))
y_test = pd.read_parquet(os.path.join(DATA_DIR, 'y_test.parquet')).squeeze()

# Corrigir outliers de idade (consistente com o pré-processamento dos modelos)
X_test = X_test.copy()
X_test.loc[X_test['age_years'] > 120, 'age_years'] = np.nan

# Base de fairness: variáveis de estratificação + target
df_test = X_test[['age_years', 'CS_SEXO', 'CS_RACA', 'CS_ESCOL_N', 'CS_GESTANT', 'SG_UF']].copy()
df_test['y_true'] = y_test.values

# Carregar predições de cada modelo (alinhadas por posição com X_test)
modelos_ok = []
for arq, label in LABEL_MAP.items():
    pred_path = os.path.join(OUTPUT_MET, f'{arq}_predicoes.parquet')
    if not os.path.exists(pred_path):
        print(f'AVISO: predições não encontradas — {label}')
        continue
    preds = pd.read_parquet(pred_path)
    df_test[f'proba_{arq}'] = preds['y_proba'].values
    modelos_ok.append(label)

# Dicionário: rótulo → coluna de probabilidade
MODELO_COLS = {label: f'proba_{arq}'
               for arq, label in LABEL_MAP.items()
               if f'proba_{arq}' in df_test.columns}

print(f'Conjunto de teste: {len(df_test):,} registros')
print(f'Eventos (óbitos): {int(df_test["y_true"].sum()):,} ({df_test["y_true"].mean()*100:.2f}%)')
print(f'Modelos carregados: {len(MODELO_COLS)} / {len(LABEL_MAP)}')
df_test.head(3)

In [ ]:
def calcular_secao(df, grupos_col, varname, n_boot=N_BOOT):
    """
    Bootstrap não paramétrico para todos os modelos em cada subgrupo.
    Índices de reamostragem compartilhados entre modelos no mesmo subgrupo.
    Subgrupos com menos de MIN_EVENTOS eventos são excluídos com aviso.
    """
    subgrupos = sorted(df[grupos_col].dropna().unique())
    rows = []

    for sg in subgrupos:
        mask    = df[grupos_col] == sg
        subset  = df[mask].reset_index(drop=True)
        n_total   = len(subset)
        n_eventos = int(subset['y_true'].sum())

        if n_eventos < MIN_EVENTOS:
            print(f'  [EXCLUÍDO] {varname} = {sg!r}: apenas {n_eventos} eventos (< {MIN_EVENTOS})')
            continue

        print(f'  Subgrupo {sg!r}: n={n_total:,}, eventos={n_eventos}', end=' ... ')

        # Índices bootstrap: gerados uma vez, compartilhados entre modelos
        rng      = np.random.RandomState(RANDOM_STATE)
        idx_boot = rng.randint(0, n_total, (n_boot, n_total))
        y_true   = subset['y_true'].values

        for label, col in MODELO_COLS.items():
            if col not in subset.columns:
                continue
            y_proba = subset[col].values

            boot = {'auprc': [], 'roc': [], 'sens': [], 'esp': []}
            for idx in idx_boot:
                yt, yp = y_true[idx], y_proba[idx]
                if yt.sum() < 2 or (yt == 0).sum() < 2:
                    continue
                yb = (yp >= THRESHOLD).astype(int)
                tp = ((yt == 1) & (yb == 1)).sum()
                fp = ((yt == 0) & (yb == 1)).sum()
                fn = ((yt == 1) & (yb == 0)).sum()
                tn = ((yt == 0) & (yb == 0)).sum()
                boot['auprc'].append(average_precision_score(yt, yp))
                boot['roc'].append(roc_auc_score(yt, yp))
                boot['sens'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
                boot['esp'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)

            def ci(v): return np.percentile(v, 2.5), np.mean(v), np.percentile(v, 97.5)
            la, ma, ha = ci(boot['auprc'])
            lr, mr, hr = ci(boot['roc'])
            ls, ms, hs = ci(boot['sens'])
            le, me, he = ci(boot['esp'])

            rows.append({
                'subgrupo':   str(sg),
                'modelo':     label,
                'n_total':    n_total,
                'n_eventos':  n_eventos,
                'auprc':          ma, 'auprc_ci_low':   la, 'auprc_ci_high':   ha,
                'roc_auc':        mr, 'roc_auc_ci_low': lr, 'roc_auc_ci_high': hr,
                'sensibilidade':  ms, 'sens_ci_low':    ls, 'sens_ci_high':    hs,
                'especificidade': me, 'esp_ci_low':     le, 'esp_ci_high':     he,
            })
        print('OK')

    return pd.DataFrame(rows)


def tabela_ic(df_sec, metrica='auprc'):
    """Tabela formatada: linhas=modelos, colunas=subgrupos, valor= 'val (lo–hi)'."""
    # calcular_secao usa nomes abreviados para sens/esp — mapeamento explícito
    CI_MAP = {
        'auprc':          ('auprc_ci_low',   'auprc_ci_high'),
        'roc_auc':        ('roc_auc_ci_low', 'roc_auc_ci_high'),
        'sensibilidade':  ('sens_ci_low',    'sens_ci_high'),
        'especificidade': ('esp_ci_low',     'esp_ci_high'),
    }
    ci_lo, ci_hi = CI_MAP.get(metrica, (f'{metrica}_ci_low', f'{metrica}_ci_high'))
    modelos   = list(MODELO_COLS.keys())
    subgrupos = sorted(df_sec['subgrupo'].unique())
    rows = {}
    for mod in modelos:
        sub = df_sec[df_sec['modelo'] == mod].set_index('subgrupo')
        row = {}
        for sg in subgrupos:
            if sg in sub.index:
                v, lo, hi = sub.loc[sg, metrica], sub.loc[sg, ci_lo], sub.loc[sg, ci_hi]
                row[sg] = f'{v:.3f} ({lo:.3f}–{hi:.3f})'
            else:
                row[sg] = '—'
        rows[mod] = row
    return pd.DataFrame(rows).T


def grafico_auprc(df_sec, titulo, filename):
    """Barras agrupadas: subgrupos no eixo-x, cores por modelo, IC 95% como error bar."""
    subgrupos = sorted(df_sec['subgrupo'].unique())
    modelos   = list(MODELO_COLS.keys())
    n_sg, n_mod = len(subgrupos), len(modelos)
    width = 0.8 / n_mod
    x = np.arange(n_sg)
    cmap = plt.get_cmap('tab20')

    fig, ax = plt.subplots(figsize=(max(11, n_sg * 1.8), 5))
    for i, mod in enumerate(modelos):
        sub  = df_sec[df_sec['modelo'] == mod].set_index('subgrupo')
        vals = [sub.loc[sg, 'auprc']          if sg in sub.index else np.nan for sg in subgrupos]
        lo   = [sub.loc[sg, 'auprc_ci_low']   if sg in sub.index else np.nan for sg in subgrupos]
        hi   = [sub.loc[sg, 'auprc_ci_high']  if sg in sub.index else np.nan for sg in subgrupos]
        err  = [[v - l if not (np.isnan(v) or np.isnan(l)) else 0 for v, l in zip(vals, lo)],
                [h - v if not (np.isnan(v) or np.isnan(h)) else 0 for h, v in zip(hi, vals)]]
        offset = (i - n_mod / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.9, label=mod,
               color=cmap(i / n_mod), alpha=0.82,
               yerr=err, error_kw={'elinewidth': 0.7, 'capsize': 1.5, 'alpha': 0.6})

    ax.set_xticks(x)
    ax.set_xticklabels(subgrupos, rotation=18, ha='right', fontsize=9)
    ax.set_ylabel('AUPRC', fontsize=10)
    ylim_hi = min(1.0, df_sec['auprc_ci_high'].max() * 1.18)
    ax.set_ylim(max(0, df_sec['auprc'].min() * 0.85), ylim_hi)
    ax.set_title(titulo, fontweight='bold', fontsize=11)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7.5,
              title='Modelo', title_fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_PLT, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gráfico salvo: {path}')


def alertas_disparidade(df_sec, varname, limiar=0.05):
    """Identifica modelos com ΔAUPRC > limiar entre melhor e pior subgrupo."""
    alertas = []
    for mod, grp in df_sec.groupby('modelo'):
        amax = grp['auprc'].max(); amin = grp['auprc'].min()
        disp = amax - amin
        sg_max = grp.loc[grp['auprc'].idxmax(), 'subgrupo']
        sg_min = grp.loc[grp['auprc'].idxmin(), 'subgrupo']
        alertas.append({
            'modelo': mod, 'variavel': varname,
            'auprc_max': amax, 'subg_max': sg_max,
            'auprc_min': amin, 'subg_min': sg_min,
            'disparidade': disp,
            'flag': 'ALERTA' if disp > limiar else 'OK',
        })
    df_al = pd.DataFrame(alertas).sort_values('disparidade', ascending=False)
    flagged = df_al[df_al['flag'] == 'ALERTA']
    print(f'Limiar ΔAUPRC > {limiar} | {len(flagged)} modelo(s) com alerta:')
    for _, r in flagged.iterrows():
        print(f"  ⚠️  {r['modelo']}: ΔAUPRC = {r['disparidade']:.4f}  "
              f"[{r['subg_max']} ({r['auprc_max']:.3f}) vs {r['subg_min']} ({r['auprc_min']:.3f})]")
    if flagged.empty:
        print('  Nenhum modelo excede o limiar.')
    return df_al

# Dicionário global para agregar disparidades na seção final
_disparidades = {}

print('Funções auxiliares definidas.')

In [ ]:
# Criar faixas etárias
bins_idade   = [-np.inf, 15, 60, np.inf]
labels_idade = ['<15 anos', '15-59 anos', '>=60 anos']
df_test['faixa_etaria'] = pd.cut(df_test['age_years'], bins=bins_idade,
                                  labels=labels_idade, right=False)

# Distribuição e prevalência por faixa
tab_idade = (df_test
    .groupby('faixa_etaria', observed=True)
    .agg(n=('y_true', 'count'), eventos=('y_true', 'sum'), taxa=('y_true', 'mean'))
    .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
    .round({'taxa': 4}))
display(tab_idade)

In [ ]:
# Bootstrap por faixa etária × modelo
# Tempo estimado: ~3–6 minutos dependendo do hardware
print('Calculando métricas por faixa etária...')
df_sec_idade = calcular_secao(df_test, 'faixa_etaria', 'faixa_etaria')
print(f'\nConcluído: {len(df_sec_idade)} linhas de resultados.')

In [ ]:
# Tabela consolidada: AUPRC com IC 95%
print('=== AUPRC (média bootstrap ± IC 95%) por Faixa Etária ===\n')
display(tabela_ic(df_sec_idade, 'auprc'))

# Tabela completa — todas as métricas
print('\n=== Tabela completa (todas as métricas) ===')
cols_show = ['subgrupo','modelo','n_total','n_eventos',
             'auprc','auprc_ci_low','auprc_ci_high',
             'roc_auc','roc_auc_ci_low','roc_auc_ci_high',
             'sensibilidade','sens_ci_low','sens_ci_high',
             'especificidade','esp_ci_low','esp_ci_high']
display(df_sec_idade[cols_show].round(4).sort_values(['subgrupo','modelo'])
        .reset_index(drop=True))

In [ ]:
# Gráfico de barras agrupadas — AUPRC por faixa etária
grafico_auprc(df_sec_idade,
              'AUPRC por Faixa Etária — todos os modelos (IC 95%)',
              'fairness_01_faixa_etaria_auprc.png')

In [ ]:
# Identificar disparidades > 0.05
df_disp_idade = alertas_disparidade(df_sec_idade, 'Faixa Etária')
_disparidades['Faixa Etária'] = df_disp_idade
display(df_disp_idade[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# Criar grupos de sexo
df_test['sexo_grupo'] = df_test['CS_SEXO'].map({'F': 'Feminino', 'M': 'Masculino'})

tab_sexo = (df_test
    .groupby('sexo_grupo')
    .agg(n=('y_true','count'), eventos=('y_true','sum'), taxa=('y_true','mean'))
    .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
    .round({'taxa': 4}))
display(tab_sexo)

In [ ]:
print('Calculando métricas por sexo...')
df_sec_sexo = calcular_secao(df_test, 'sexo_grupo', 'CS_SEXO')
print(f'\nConcluído: {len(df_sec_sexo)} linhas de resultados.')

In [ ]:
print('=== AUPRC (IC 95%) por Sexo ===\n')
display(tabela_ic(df_sec_sexo, 'auprc'))

print('\n=== Sensibilidade (IC 95%) por Sexo ===')
display(tabela_ic(df_sec_sexo, 'sensibilidade'))

print('\n=== Tabela completa ===')
display(df_sec_sexo[cols_show].round(4).sort_values(['subgrupo','modelo']).reset_index(drop=True))

In [ ]:
grafico_auprc(df_sec_sexo,
              'AUPRC por Sexo — todos os modelos (IC 95%)',
              'fairness_02_sexo_auprc.png')

In [ ]:
df_disp_sexo = alertas_disparidade(df_sec_sexo, 'Sexo')
_disparidades['Sexo'] = df_disp_sexo
display(df_disp_sexo[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# Mapear CS_RACA para rótulos legíveis
raca_map = {1.0: 'Branca', 2.0: 'Preta', 3.0: 'Amarela', 4.0: 'Parda', 5.0: 'Indigena'}
df_test['raca_grupo'] = df_test['CS_RACA'].map(raca_map)

tab_raca = (df_test
    .groupby('raca_grupo')
    .agg(n=('y_true','count'), eventos=('y_true','sum'), taxa=('y_true','mean'))
    .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
    .round({'taxa': 4}))
print('Distribuição por raça/cor (subgrupos com <30 eventos serão excluídos):')
display(tab_raca)

In [ ]:
print('Calculando métricas por raça/cor...')
df_sec_raca = calcular_secao(df_test, 'raca_grupo', 'CS_RACA')
print(f'\nConcluído: {len(df_sec_raca)} linhas de resultados.')

In [ ]:
print('=== AUPRC (IC 95%) por Raça/Cor ===\n')
display(tabela_ic(df_sec_raca, 'auprc'))

print('\n=== Sensibilidade (IC 95%) por Raça/Cor ===')
display(tabela_ic(df_sec_raca, 'sensibilidade'))

print('\n=== Tabela completa ===')
display(df_sec_raca[cols_show].round(4).sort_values(['subgrupo','modelo']).reset_index(drop=True))

In [ ]:
grafico_auprc(df_sec_raca,
              'AUPRC por Raça/Cor — todos os modelos (IC 95%)',
              'fairness_03_raca_auprc.png')

In [ ]:
df_disp_raca = alertas_disparidade(df_sec_raca, 'Raca/Cor')
_disparidades['Raça/Cor'] = df_disp_raca
display(df_disp_raca[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# Agrupar CS_ESCOL_N em faixas de maior poder estatístico
escol_grupo_map = {
    0.0:  'Sem/Fund.I',   # Sem escolaridade a Fund. I completo
    1.0:  'Sem/Fund.I',
    2.0:  'Sem/Fund.I',
    3.0:  'Fund.II',      # Fund. II incompleto e completo
    4.0:  'Fund.II',
    5.0:  'Médio',        # Ensino Médio incompleto e completo
    6.0:  'Médio',
    7.0:  'Superior',     # Superior incompleto e completo
    8.0:  'Superior',
    10.0: 'N/A (<8 anos)',# Não se aplica por idade
}
df_test['escol_grupo'] = df_test['CS_ESCOL_N'].map(escol_grupo_map)

tab_escol = (df_test
    .groupby('escol_grupo')
    .agg(n=('y_true','count'), eventos=('y_true','sum'), taxa=('y_true','mean'))
    .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
    .round({'taxa': 4}))
print(f'Registros com escolaridade ausente: {df_test["CS_ESCOL_N"].isna().sum():,} '
      f'({df_test["CS_ESCOL_N"].isna().mean()*100:.1f}%)')
display(tab_escol)

In [ ]:
print('Calculando métricas por escolaridade...')
df_sec_escol = calcular_secao(df_test, 'escol_grupo', 'CS_ESCOL_N')
print(f'\nConcluído: {len(df_sec_escol)} linhas de resultados.')

In [ ]:
print('=== AUPRC (IC 95%) por Escolaridade ===\n')
display(tabela_ic(df_sec_escol, 'auprc'))

print('\n=== Especificidade (IC 95%) por Escolaridade ===')
display(tabela_ic(df_sec_escol, 'especificidade'))

print('\n=== Tabela completa ===')
display(df_sec_escol[cols_show].round(4).sort_values(['subgrupo','modelo']).reset_index(drop=True))

In [ ]:
grafico_auprc(df_sec_escol,
              'AUPRC por Escolaridade — todos os modelos (IC 95%)',
              'fairness_04_escolaridade_auprc.png')

In [ ]:
df_disp_escol = alertas_disparidade(df_sec_escol, 'Escolaridade')
_disparidades['Escolaridade'] = df_disp_escol
display(df_disp_escol[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# # Agrupar CS_GESTANT
# gestant_map = {
#     1.0: 'Gestante',         # 1° trimestre
#     2.0: 'Gestante',         # 2° trimestre
#     3.0: 'Gestante',         # 3° trimestre
#     4.0: 'Gestante',         # trimestre ignorado
#     5.0: 'Não gestante',     # não gestante
#     6.0: 'Não se aplica',    # masculino / < 9 anos
# }
# df_test['gestant_grupo'] = df_test['CS_GESTANT'].map(gestant_map)

# tab_gestant = (df_test
#     .groupby('gestant_grupo')
#     .agg(n=('y_true','count'), eventos=('y_true','sum'), taxa=('y_true','mean'))
#     .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
#     .round({'taxa': 4}))
# display(tab_gestant)
# print('\nNota: grupos com <30 eventos serão excluídos automaticamente na análise.')

In [ ]:
# print('Calculando métricas por situação de gestação...')
# df_sec_gestant = calcular_secao(df_test, 'gestant_grupo', 'CS_GESTANT')
# print(f'\nConcluído: {len(df_sec_gestant)} linhas de resultados.')

In [ ]:
# print('=== AUPRC (IC 95%) por Gestação ===\n')
# display(tabela_ic(df_sec_gestant, 'auprc'))

# print('\n=== Sensibilidade (IC 95%) por Gestação ===')
# display(tabela_ic(df_sec_gestant, 'sensibilidade'))

# print('\n=== Tabela completa ===')
# display(df_sec_gestant[cols_show].round(4).sort_values(['subgrupo','modelo']).reset_index(drop=True))

In [ ]:
# grafico_auprc(df_sec_gestant,
#               'AUPRC por Situação de Gestação — todos os modelos (IC 95%)',
#               'fairness_05_gestacao_auprc.png')

In [ ]:
# df_disp_gestant = alertas_disparidade(df_sec_gestant, 'Gestação')
# _disparidades['Gestação'] = df_disp_gestant
# display(df_disp_gestant[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# Mapear código IBGE de UF para macrorregião
regiao_map = {
    '11':'Norte', '12':'Norte', '13':'Norte', '14':'Norte',
    '15':'Norte', '16':'Norte', '17':'Norte',
    '21':'Nordeste', '22':'Nordeste', '23':'Nordeste', '24':'Nordeste',
    '25':'Nordeste', '26':'Nordeste', '27':'Nordeste', '28':'Nordeste', '29':'Nordeste',
    '31':'Sudeste', '32':'Sudeste', '33':'Sudeste', '35':'Sudeste',
    '41':'Sul', '42':'Sul', '43':'Sul',
    '50':'Centro-Oeste', '51':'Centro-Oeste', '52':'Centro-Oeste', '53':'Centro-Oeste',
}
df_test['regiao'] = df_test['SG_UF'].map(regiao_map)

tab_regiao = (df_test
    .groupby('regiao')
    .agg(n=('y_true','count'), eventos=('y_true','sum'), taxa=('y_true','mean'))
    .assign(pct_total=lambda d: (d['n'] / d['n'].sum() * 100).round(1))
    .round({'taxa': 4}))
display(tab_regiao)

In [ ]:
print('Calculando métricas por região geográfica...')
df_sec_regiao = calcular_secao(df_test, 'regiao', 'SG_UF_regiao')
print(f'\nConcluído: {len(df_sec_regiao)} linhas de resultados.')

In [ ]:
print('=== AUPRC (IC 95%) por Região ===\n')
display(tabela_ic(df_sec_regiao, 'auprc'))

print('\n=== Sensibilidade (IC 95%) por Região ===')
display(tabela_ic(df_sec_regiao, 'sensibilidade'))

print('\n=== Tabela completa ===')
display(df_sec_regiao[cols_show].round(4).sort_values(['subgrupo','modelo']).reset_index(drop=True))

In [ ]:
grafico_auprc(df_sec_regiao,
              'AUPRC por Região Geográfica — todos os modelos (IC 95%)',
              'fairness_06_regiao_auprc.png')

In [ ]:
df_disp_regiao = alertas_disparidade(df_sec_regiao, 'Região')
_disparidades['Região'] = df_disp_regiao
display(df_disp_regiao[['modelo','disparidade','auprc_max','subg_max','auprc_min','subg_min','flag']].round(4))

In [ ]:
# Tabela resumo: maior disparidade por variável × modelo
rows_resumo = []
for var, df_disp in _disparidades.items():
    for _, r in df_disp.iterrows():
        rows_resumo.append({
            'variavel':   var,
            'modelo':     r['modelo'],
            'disparidade': r['disparidade'],
            'flag':        r['flag'],
        })

df_resumo = pd.DataFrame(rows_resumo)

# Pivotar: linhas=modelo, colunas=variável
pivot_disp = df_resumo.pivot_table(index='modelo', columns='variavel',
                                    values='disparidade', aggfunc='first')

print('=== Disparidade AUPRC (Δmax−min) por Modelo × Variável ===\n')
display(pivot_disp.round(4).style
    .background_gradient(cmap='YlOrRd', vmin=0, vmax=0.15)
    .format('{:.4f}')
    .set_caption('Vermelho = maior disparidade | Limiar de alerta: 0.05'))

# Contar alertas por variável
print('\n=== Contagem de alertas (ΔAUPRC > 0.05) por variável ===')
alertas_por_var = (df_resumo[df_resumo['flag']=='ALERTA']
    .groupby('variavel').size()
    .rename('n_modelos_alertados')
    .to_frame())
display(alertas_por_var)

# Variável com maior dispersão média
print('\n=== Variável com maior disparidade média ===')
display(df_resumo.groupby('variavel')['disparidade']
    .agg(['mean','max','min'])
    .round(4)
    .sort_values('mean', ascending=False))

In [ ]:
# Heatmap: modelos × variáveis — magnitude da disparidade AUPRC
fig, ax = plt.subplots(figsize=(11, 7))

sns.heatmap(
    pivot_disp,
    annot=True, fmt='.3f', cmap='YlOrRd',
    vmin=0, vmax=0.15, linewidths=0.4,
    ax=ax,
    cbar_kws={'label': 'ΔAUPRC (max − min subgrupo)', 'shrink': 0.8},
)



ax.set_title('Disparidade de AUPRC por Modelo × Variável de Estratificação\n'
             ,
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Variável de Estratificação', fontsize=10)
ax.set_ylabel('Modelo', fontsize=10)
plt.xticks(rotation=15, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

path_heat = os.path.join(OUTPUT_PLT, 'fairness_heatmap_disparidades.png')
plt.savefig(path_heat, dpi=150, bbox_inches='tight')
plt.show()
print(f'Heatmap salvo: {path_heat}')

In [ ]:
# Salvar tabela resumo como CSV
csv_path = os.path.join(OUTPUT_PLT, '..', 'metricas', 'fairness_disparidades_resumo.csv')
df_resumo.to_csv(csv_path, index=False)
print(f'Resumo salvo: {csv_path}')

# Salvar resultado completo de cada seção
for var, df_disp in _disparidades.items():
    slug = var.lower().replace('/', '_').replace(' ', '_')
    p = os.path.join(OUTPUT_PLT, '..', 'metricas', f'fairness_{slug}_detalhado.csv')
    df_disp.to_csv(p, index=False)
print('Resultados detalhados por variável salvos.')